In [68]:
import pandas as pd
pd.set_option('display.max_columns', None)
from pybaseball import statcast, playerid_reverse_lookup, team_game_logs

In [2]:
mlb_teams = [
    "ARI", "ATL", "BAL", "BOS", "CHC", "CWS", "CIN", "CLE", "COL", "DET",
    "HOU", "KC", "LAA", "LAD", "MIA", "MIL", "MIN", "NYM", "NYY", "OAK",
    "PHI", "PIT", "SD", "SF", "SEA", "STL", "TB", "TEX", "TOR", "WSH"
]

In [6]:
team_df_lst = []
game_num_dict = {}

# get data from each team
for team in mlb_teams:
    # get regular season data
    reg_df = statcast(start_dt="2025-03-27", end_dt="2025-09-28", team=team, verbose=False)

    # check LAD or CHC
    if team == "LAD" or team == "CHC":
        # get Tokyo series
        tokyo_df = statcast(start_dt="2025-03-18", end_dt="2025-03-19", team=team, verbose=False)

        # combine with regular season data
        reg_df = pd.concat([reg_df, tokyo_df], ignore_index=True)
    
    # check game number
    game_num = len(reg_df["game_pk"].unique())
    game_num_dict[team] = game_num

    # append to list
    team_df_lst.append(reg_df)

/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
  0%|          | 0/186 [00:00<?, ?it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explic

In [8]:
# combine all the data
df = pd.concat(team_df_lst, ignore_index=True)

In [79]:
def convert_to_PA(df):
    """
    Convert pitch level dataframe to PA level.
    The df should only passed in one game data.
    """

    # copy the dataframe
    df = df.copy()
    df["outs"] = df["outs_when_up"]

    # keep last pitch data of each at bat
    df = df.sort_values(['game_pk', 'at_bat_number', 'pitch_number'])

    # change runner on base format
    df[["on_3b", "on_2b", "on_1b"]] = df[["on_3b", "on_2b", "on_1b"]].fillna(0) # if no base runner = 0
    for col in ["on_3b", "on_2b", "on_1b"]: # if have base runner = 1
        df.loc[df[col] != 0, col] = 1

    # get needed post feature
    # sort df
    df = df.sort_values(["game_pk", "at_bat_number", "pitch_number"])

    # shift each game
    for col in ["on_3b", "on_2b", "on_1b", "inning", "inning_topbot", "outs"]:
        df[f"post_{col}"] = df.groupby("game_pk")[col].shift(-1)

    # fill last AB as -1
    df[["post_on_3b", "post_on_2b", "post_on_1b"]] = df[["post_on_3b", "post_on_2b", "post_on_1b"]].fillna(-1)
    
    # # only filter first and last pitch of PA
    # # first pitch of each PA
    # first = (
    #     df.groupby(['game_pk', 'at_bat_number'], as_index=False)
    #       .nth(0)          # <-- ALWAYS works, even if groups have 1 row
    #       .copy()
    # )
    # first['pitch_role'] = 'first'

    # # last pitch of each PA
    # last = (
    #     df.groupby(['game_pk', 'at_bat_number'], as_index=False)
    #       .nth(-1)         # <-- ALWAYS works
    #       .copy()
    # )
    # last['pitch_role'] = 'last'

    # # combine them
    # df = pd.concat([first, last], ignore_index=True)

    # # selected essential columns
    # df = df.sort_values(by=["game_pk", "at_bat_number", "pitch_number"])

    # compute RBI
    df["RBI"] = df["post_bat_score"] - df["bat_score"]

    # we only need RBI not equal to zero data
    df = df[df["RBI"] != 0]

    # we only need to keep last pitch
    # df = df[df["pitch_role"] == "last"]

    # match hitter and pitcher name
    df["batter_name"] = df["batter"].apply(lookup_name)
    df["pitcher_name"] = df["pitcher"].apply(lookup_name)

    # select essential columns
    df = df[["at_bat_number", "pitch_number", "game_pk", "home_team", "away_team", "inning", "post_inning",
             "inning_topbot", "post_inning_topbot", "outs", "post_outs",
              "pitcher", "pitcher_name",
             "batter", "batter_name", "on_3b", "on_2b", "on_1b", "post_on_3b", "post_on_2b", "post_on_1b",
             "home_score", "post_home_score", "away_score", "post_away_score", "bat_score", "post_bat_score", "RBI"]]
    
    # drop same game, at bat, and pitch number
    print(len(df))
    df = df.drop_duplicates(subset=["game_pk", "at_bat_number", "pitch_number"])
    print(len(df))
    
    # sort df
    df = df.sort_values(["game_pk", "at_bat_number", "pitch_number"])
    
    # fill or na as -1
    df = df.fillna(-1)

    return df

def lookup_name(id):
    """
    Return given player id name.
    """

    # get name dataframe
    name = playerid_reverse_lookup([id])

    # get first and last name
    first_name = name["name_first"].iloc[0]
    last_name = name["name_last"].iloc[0]

    # combine 
    full_name = f"{first_name} {last_name}"

    return full_name

In [80]:
# convert to where only has RBI PA
all_game_df = convert_to_PA(df)

16626
16626


In [81]:
# change Top to 0, bottom to 0.5
all_game_df["inning_topbot"] = all_game_df["inning_topbot"].replace({"Top": 0, "Bot": 0.5})
all_game_df["post_inning_topbot"] = all_game_df["post_inning_topbot"].replace({"Top": 0, "Bot": 0.5})

# add game difference
all_game_df["score_diff"] = all_game_df["home_score"] - all_game_df["away_score"]
all_game_df["post_score_diff"] = all_game_df["post_home_score"] - all_game_df["post_away_score"]

# save as parquet
all_game_df.to_parquet("PA_level_data.parquet")

/var/folders/r1/fdb0tbq57qj3ww3lz_lh0xf00000gn/T/ipykernel_37514/1720093726.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_game_df["inning_topbot"] = all_game_df["inning_topbot"].replace({"Top": 0, "Bot": 0.5})
/var/folders/r1/fdb0tbq57qj3ww3lz_lh0xf00000gn/T/ipykernel_37514/1720093726.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_game_df["post_inning_topbot"] = all_game_df["post_inning_topbot"].replace({"Top": 0, "Bot": 0.5})
